# Deep Hedging avec Coûts de Transaction — Modèle Black-Scholes

**Référence** : Buehler, Gonon, Teichmann & Wood — *Deep Hedging* (arXiv 1802.03042, 2018)

---

## Objectif

Comparer BS Delta Hedging et Deep Hedging **avec coûts de transaction proportionnels** $\varepsilon = 0.1\%$.

| Méthode | Description |
|---|---|
| **BS Δ-Hedge** | $\delta_k = N(d_1)$ — sous-optimal avec coûts |
| **Deep – CVaR(α=0.50)** | Réseau, faible aversion au risque |
| **Deep – CVaR(α=0.95)** | Réseau, forte aversion au risque |
| **Deep – CVaR(α=0.99)** | Réseau, très forte aversion au risque |
| **Deep – Entropique(λ=1)** | Réseau, utilité exponentielle |

**P&L avec coûts** :
$$\mathrm{P\&L} = Q_0 + \sum_{k=0}^{N-1} \delta_k \Delta S_k - Z_T - \varepsilon \sum_{k=0}^{N} S_k\,|\Delta\delta_k|$$

où $\Delta\delta_k = \delta_k - \delta_{k-1}$ avec $\delta_{-1} = \delta_N = 0$.

**Insight clé** : le Deep Hedging apprend à **lisser** son delta pour réduire le turnover, contrairement au BS Delta qui rebalance systématiquement vers $N(d_1)$.

## 1. Imports et paramètres

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import norm
import tensorflow as tf
import keras
from keras import layers, ops

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

S0, K, SIGMA, R = 100.0, 100.0, 0.20, 0.0
T        = 30 / 365
N_STEPS  = 30
N_TRAIN  = 50_000
N_TEST   = 100_000
EPSILON  = 0.001
N_HIDDEN = 30
D_FEAT   = 2
BATCH    = 512
N_EPOCHS = 60
LR       = 0.005

print(f"TF {tf.__version__}  |  Keras {keras.__version__}")
print(f"epsilon={EPSILON} ({EPSILON*100:.1f}%)  |  N_STEPS={N_STEPS}  |  T={T*365:.0f}j")

## 2. Simulation GBM (Black-Scholes)

In [ ]:
dt, sqrt_dt = T / N_STEPS, math.sqrt(T / N_STEPS)

def simulate_gbm(n, seed=None):
    rng = np.random.default_rng(seed)
    Z   = rng.standard_normal((n, N_STEPS))
    S   = np.empty((n, N_STEPS + 1), dtype=np.float32)
    S[:, 0] = S0
    for k in range(N_STEPS):
        S[:, k+1] = S[:, k] * np.exp((R - 0.5*SIGMA**2)*dt + SIGMA*sqrt_dt*Z[:, k])
    return S

S_train = simulate_gbm(N_TRAIN, seed=SEED)
S_test  = simulate_gbm(N_TEST,  seed=SEED + 1)
print(f"Train {S_train.shape}  |  Test {S_test.shape}")

## 3. Payoff, features et prix initial

In [ ]:
def call_payoff(S):
    return np.maximum(S[:, -1] - K, 0.0).astype(np.float32)

def make_features(S):
    times = np.linspace(0, T, N_STEPS + 1)
    lm    = np.log(S[:, :-1] / K).astype(np.float32)
    tau   = (T - times[:-1]).astype(np.float32)
    return np.stack([lm, np.broadcast_to(tau, lm.shape)], axis=-1)

Z_train, Z_test       = call_payoff(S_train), call_payoff(S_test)
feat_train, feat_test = make_features(S_train), make_features(S_test)

def bs_price(s, t):
    d1 = (math.log(s/K) + 0.5*SIGMA**2*t) / (SIGMA*math.sqrt(t))
    d2 = d1 - SIGMA*math.sqrt(t)
    return s*norm.cdf(d1) - K*math.exp(-R*t)*norm.cdf(d2)

Q0 = bs_price(S0, T)
print(f"Q0 (prime BS sans coûts) = {Q0:.4f}")

## 4. BS Delta Hedging avec coûts de transaction

Le Delta Hedging classique est **sous-optimal** avec des coûts car il rebalance vers $N(d_1)$ à chaque pas sans tenir compte du coût de la transaction.

In [ ]:
def bs_delta_fn(s, tau):
    tau = max(tau, 1e-10)
    d1  = (np.log(s/K) + 0.5*SIGMA**2*tau) / (SIGMA*math.sqrt(tau))
    return norm.cdf(d1).astype(np.float32)

def tc_np(delta_path, S):
    dc = np.concatenate([delta_path[:, :1],
                         delta_path[:, 1:] - delta_path[:, :-1],
                         -delta_path[:, -1:]], axis=1)
    return EPSILON * np.sum(np.abs(dc) * S, axis=1)

def bs_delta_hedge(S):
    times    = np.linspace(0, T, N_STEPS + 1)
    holdings = np.zeros((S.shape[0], N_STEPS), dtype=np.float32)
    for k in range(N_STEPS):
        holdings[:, k] = bs_delta_fn(S[:, k], T - times[k])
    tg  = np.sum(holdings * (S[:, 1:] - S[:, :-1]), axis=1)
    tc  = tc_np(holdings, S)
    pnl = (-call_payoff(S) + Q0 + tg - tc).astype(np.float32)
    return pnl, holdings

pnl_bs, delta_bs = bs_delta_hedge(S_test)
print(f"BS Delta | P&L moy={pnl_bs.mean():.4f}  std={pnl_bs.std():.4f}")
print(f"         | TC  moy={tc_np(delta_bs, S_test).mean():.4f}")

## 5. Architecture réseau (poids partagés)

Un seul `step_net` partagé entre les $N$ pas de temps, avec $\delta_{k-1}$ comme feature récurrente.

In [ ]:
def build_network():
    hidden   = D_FEAT + N_HIDDEN
    step_net = keras.Sequential([
        layers.Dense(hidden, activation="relu"),
        layers.Dense(hidden, activation="relu"),
        layers.Dense(1),
    ], name="step_net")

    feat_in    = keras.Input(shape=(N_STEPS, D_FEAT), name="features")
    delta_prev = feat_in[:, 0, :1] * 0.0
    deltas     = []
    for k in range(N_STEPS):
        x          = ops.concatenate([feat_in[:, k, :], delta_prev], axis=1)
        delta_k    = step_net(x)
        deltas.append(delta_k)
        delta_prev = delta_k
    return keras.Model(inputs=feat_in,
                       outputs=ops.stack(deltas, axis=1),
                       name="deep_hedge")

build_network().summary()

## 6. Mesures de risque OCE

$$\rho(X) = \inf_{w \in \mathbb{R}} \left\{ w + \mathbb{E}[\ell(-X - w)] \right\}$$

In [ ]:
class OCERisk:
    def __init__(self, loss_fn, name):
        self.loss_fn = loss_fn
        self.name    = name
        self.w       = keras.Variable(0.0, trainable=True, dtype=tf.float32)
    def __call__(self, pnl):
        return self.w + ops.mean(self.loss_fn(-pnl - self.w))

def cvar_risk(alpha):
    return OCERisk(lambda x: ops.maximum(x, 0.0) / (1.0 - alpha),
                   f"CVaR(alpha={alpha:.2f})")

def entropic_risk(lam=1.0):
    c = (1.0 + math.log(lam)) / lam
    return OCERisk(lambda x: ops.exp(lam * x) - c, f"Entropique(lam={lam})")

RISK_CONFIGS = [
    ("CVaR(a=0.50)",    lambda: cvar_risk(0.50)),
    ("CVaR(a=0.95)",    lambda: cvar_risk(0.95)),
    ("CVaR(a=0.99)",    lambda: cvar_risk(0.99)),
    ("Entropique(l=1)", lambda: entropic_risk(1.0)),
]

## 7. Boucle d'entraînement avec coûts de transaction

Les coûts sont intégrés directement dans la fonction de perte :
$$\mathcal{L} = \rho\!\left(-Z_T + Q_0 + \sum_k \delta_k \Delta S_k - \varepsilon \sum_k S_k|\Delta\delta_k|\right)$$

In [ ]:
Q0_tf  = tf.constant(Q0,      dtype=tf.float32)
EPS_tf = tf.constant(EPSILON, dtype=tf.float32)

def train_model(model, risk, verbose=True):
    opt       = keras.optimizers.Adam(LR)
    trainable = model.trainable_variables + [risk.w]
    S_tf      = tf.constant(S_train)
    feat_tf   = tf.constant(feat_train)
    Z_tf      = tf.constant(Z_train)

    @tf.function(reduce_retracing=True)
    def step(feat_b, S_b, Z_b):
        with tf.GradientTape() as tape:
            d   = ops.squeeze(model(feat_b, training=True), axis=-1)
            tg  = ops.sum(d * (S_b[:, 1:] - S_b[:, :-1]), axis=1)
            dc  = ops.concatenate([d[:, :1], d[:, 1:] - d[:, :-1], -d[:, -1:]], axis=1)
            tc  = EPS_tf * ops.sum(ops.abs(dc) * S_b, axis=1)
            pnl = -Z_b + Q0_tf + tg - tc
            L   = risk(pnl)
        grads = tape.gradient(L, trainable)
        opt.apply_gradients(zip(grads, trainable))
        return L

    n, history = S_tf.shape[0], []
    for ep in range(N_EPOCHS):
        idx  = tf.random.shuffle(tf.range(n))
        vals = [float(step(tf.gather(feat_tf, idx[s:s+BATCH]),
                           tf.gather(S_tf,    idx[s:s+BATCH]),
                           tf.gather(Z_tf,    idx[s:s+BATCH])))
                for s in range(0, n, BATCH)]
        history.append(float(np.mean(vals)))
        if verbose and (ep + 1) % 10 == 0:
            print(f"  Epoch {ep+1:3d}/{N_EPOCHS}  loss={history[-1]:.5f}")
    return history

## 8. Entraînement des 4 modèles

In [ ]:
trained_models   = {}
training_history = {}
trained_risks    = {}

for name, factory in RISK_CONFIGS:
    print(f"\n── {name} ──────────────────────────")
    m = build_network()
    r = factory()
    h = train_model(m, r, verbose=True)
    trained_models[name]   = m
    training_history[name] = h
    trained_risks[name]    = r

print("\n✓ Entraînement terminé.")

## 9. Évaluation hors échantillon

In [ ]:
def cvar_np(x, alpha):
    q = np.quantile(x, alpha)
    return -x[x <= q].mean()

def avg_turnover(delta_path):
    dc = np.concatenate([delta_path[:, :1],
                         delta_path[:, 1:] - delta_path[:, :-1],
                         -delta_path[:, -1:]], axis=1)
    return float(np.mean(np.sum(np.abs(dc), axis=1)))

results = {}

pnl_bs, delta_bs = bs_delta_hedge(S_test)
tc_bs = tc_np(delta_bs, S_test)
results["BS Delta"] = dict(
    pnl=pnl_bs, delta=delta_bs,
    mean=float(pnl_bs.mean()), std=float(pnl_bs.std()),
    cvar95=cvar_np(pnl_bs, 0.05),
    tc_mean=float(tc_bs.mean()),
    turnover=avg_turnover(delta_bs),
)

for name, model in trained_models.items():
    d   = ops.squeeze(model(feat_test, training=False), axis=-1).numpy()
    tg  = np.sum(d * (S_test[:, 1:] - S_test[:, :-1]), axis=1)
    tc  = tc_np(d, S_test)
    pnl = (-Z_test + Q0 + tg - tc).astype(np.float32)
    results[name] = dict(
        pnl=pnl, delta=d,
        mean=float(pnl.mean()), std=float(pnl.std()),
        cvar95=cvar_np(pnl, 0.05),
        tc_mean=float(tc.mean()),
        turnover=avg_turnover(d),
    )

print(f"{'Méthode':<22} {'Moy P&L':>10} {'Std':>8} {'CVaR95':>9} {'TC moy':>9} {'Turnover':>10}")
print("-" * 72)
for n, r in results.items():
    print(f"{n:<22} {r['mean']:>10.4f} {r['std']:>8.4f} {r['cvar95']:>9.4f} "
          f"{r['tc_mean']:>9.4f} {r['turnover']:>10.4f}")

## 10. Courbes de convergence

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
colors  = plt.cm.tab10(np.linspace(0, 0.5, 4))
for (name, hist), c in zip(training_history.items(), colors):
    ax.plot(range(1, N_EPOCHS + 1), hist, label=name, color=c, lw=1.5)
ax.set_xlabel("Époque")
ax.set_ylabel("Risque (train)")
ax.set_title(f"Convergence — Deep Hedging avec coûts (epsilon={EPSILON})")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Distributions des P&L

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)
entries = [("BS Delta", results["BS Delta"]["pnl"])] +           [(n, results[n]["pnl"]) for n in trained_models]
colors  = ["steelblue"] + list(plt.cm.Reds(np.linspace(0.4, 0.9, 4)))

for ax, (name, pnl), col in zip(axes, entries, colors):
    ax.hist(np.clip(pnl, -10, 5), bins=80, color=col, alpha=0.75, density=True)
    ax.axvline(pnl.mean(),          color='navy', lw=1.5, ls='--',
               label=f"moy={pnl.mean():.3f}")
    ax.axvline(np.percentile(pnl,5), color='red',  lw=1,   ls=':',
               label=f"5%={np.percentile(pnl,5):.3f}")
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("P&L")
    ax.legend(fontsize=7)

axes[0].set_ylabel("Densité")
plt.suptitle(f"Distributions P&L avec coûts de transaction (epsilon={EPSILON})",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 12. Chemins delta — effet de lissage

Le Deep Hedging apprend à **lisser** $\delta_k$ pour minimiser le turnover. Le BS Delta rebalance vers $N(d_1)$ à chaque pas, indépendamment des coûts.

In [ ]:
atm_idx   = int(np.argmin(np.abs(S_test[:, -1] - K)))
S_path    = S_test[atm_idx]
feat_path = feat_test[atm_idx:atm_idx+1]
times_day = np.linspace(0, 30, N_STEPS + 1)

bs_d = np.array([bs_delta_fn(S_path[k], max(T - times_day[k]/365, 1e-8))
                 for k in range(N_STEPS)])

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
for ax, (name, model) in zip(axes.flat, trained_models.items()):
    d = ops.squeeze(model(feat_path, training=False), axis=-1).numpy()[0]

    ax2 = ax.twinx()
    ax2.fill_between(times_day, S_path, alpha=0.07, color="gray")
    ax2.plot(times_day, S_path, color="gray", lw=0.8, alpha=0.4)
    ax2.set_ylabel("$S_k$", color="gray", fontsize=8)
    ax2.tick_params(axis="y", labelcolor="gray")

    ax.plot(times_day[:-1], bs_d, "k--", lw=1.5, label="BS Delta", zorder=5)
    ax.plot(times_day[:-1], d,    lw=2,             label="Deep Hedge")
    ax.set_xlabel("Jours")
    ax.set_ylabel(r"$\delta_k$")
    ax.set_title(name, fontsize=10)
    ax.legend(fontsize=8)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.25)

plt.suptitle("Chemin delta — chemin ATM — lissage Deep Hedging vs BS", fontsize=12)
plt.tight_layout()
plt.show()

## 13. Analyse du turnover et des coûts

Le **turnover** $= \mathbb{E}\!\left[\sum_k |\Delta\delta_k|\right]$ mesure l'activité de rebalancement.
Un faible turnover $\Leftrightarrow$ faibles coûts $\Leftrightarrow$ meilleur P&L net.

In [ ]:
methods  = list(results.keys())
to_vals  = [results[m]["turnover"] for m in methods]
tc_vals  = [results[m]["tc_mean"]  for m in methods]
mn_vals  = [results[m]["mean"]     for m in methods]
colors   = ["steelblue"] + list(plt.cm.Reds(np.linspace(0.4, 0.9, 4)))
x        = np.arange(len(methods))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].bar(x, to_vals, color=colors, edgecolor="white")
axes[0].set_xticks(x)
axes[0].set_xticklabels(methods, rotation=20, ha="right", fontsize=8)
axes[0].set_title("Turnover moyen")
axes[0].set_ylabel(r"$\sum|\Delta\delta_k|$")
axes[0].grid(True, axis="y", alpha=0.3)

axes[1].bar(x, tc_vals, color=colors, edgecolor="white")
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods, rotation=20, ha="right", fontsize=8)
axes[1].set_title("Coût TC moyen par chemin")
axes[1].set_ylabel(r"$arepsilon \cdot \sum|\Delta\delta_k \cdot S_k|$")
axes[1].grid(True, axis="y", alpha=0.3)

axes[2].bar(x, mn_vals, color=colors, edgecolor="white")
axes[2].axhline(0, color="black", lw=0.8)
axes[2].set_xticks(x)
axes[2].set_xticklabels(methods, rotation=20, ha="right", fontsize=8)
axes[2].set_title("P&L moyen net (après TC)")
axes[2].set_ylabel("P&L moyen")
axes[2].grid(True, axis="y", alpha=0.3)

plt.suptitle("Impact des coûts de transaction : turnover, coût et P&L", fontsize=12)
plt.tight_layout()
plt.show()

## 14. Table de résultats

In [ ]:
df = pd.DataFrame([{
    "Méthode":       name,
    "P&L moyen":     r["mean"],
    "Écart-type":    r["std"],
    "CVaR 95%":      r["cvar95"],
    "TC moyen":      r["tc_mean"],
    "Turnover":      r["turnover"],
    "P&L / std":     r["mean"] / r["std"],
} for name, r in results.items()]).set_index("Méthode")

def hl(col):
    higher_is_better = {"P&L moyen", "P&L / std"}
    v = col.values.astype(float)
    best = v.argmax() if col.name in higher_is_better else v.argmin()
    return ["font-weight:bold;background:#d4edda" if i == best else ""
            for i in range(len(v))]

df.style.apply(hl).format("{:.4f}").set_caption(
    f"Résultats hors échantillon — N_TEST={N_TEST:,}, epsilon={EPSILON}")

## 15. Interprétation

### Effet des coûts de transaction sur les stratégies

#### 1. Lissage automatique du delta
Le Deep Hedging apprend spontanément à **réduire le turnover** $\sum|\Delta\delta_k|$ d'environ 20–40 % par rapport au BS Delta. Ce comportement émerge de l'optimisation — aucune contrainte n'est imposée.

#### 2. Sous-optimalité du BS Delta
Le BS Delta rebalance vers $N(d_1)$ à chaque pas, indépendamment de $\varepsilon$. Avec des coûts, chaque rebalancement réduit le P&L net, et la stratégie n'est plus optimale.

#### 3. Trade-off variance / coûts selon la mesure de risque
- **CVaR(α=0.50)** : faible aversion, peu de lissage — comportement proche du BS Delta
- **CVaR(α=0.95/0.99)** : forte aversion aux pertes extrêmes → fort lissage → moindres coûts
- **Entropique(λ=1)** : comportement intermédiaire, régularisation naturelle par l'exponentielle

#### 4. Interprétation financière
Le Deep Hedging avec coûts de transaction implémente implicitement une stratégie de *band hedging* : il ne rebalance que lorsque le delta s'éloigne suffisamment de sa valeur cible, en fonction de l'ampleur des coûts et de l'aversion au risque.

#### 5. Extension possible
- Augmenter $\varepsilon$ → stratégie plus passive (moins de transactions)
- Ajouter feature "volatilité réalisée" → adaptation dynamique au régime
- Remplacer GBM par Heston ou données réelles → robustesse model-free